# Hafta 4 – 03: Gerçekten iyileşti mi? (Ölç ve iyileştir)

> Hafta 4'ün üçüncü notebook'u (Görev 1c). Notebook 01 ve 02'de yanıtı değiştirdik. "Daha iyi göründü" yetmez; **ölçeriz**.
> Bu notebook üç ölçüm aracını ve sahadan gelen geri bildirimin bir sonraki ölçümün sorusuna nasıl dönüştüğünü gösterir.

| Notebook | Soru | Ardından |
|---|---|---|
| 01_prompt_tasarimi | Modele ne söylüyoruz? | Lab 2: taslak şablonu v2 |
| 02_guvenli_yanit | Müşteriye ne gidiyor? | Lab 3: araç çıktısını maskeleme |
| **03_olc_ve_iyilestir** (bu) | Gerçekten iyileşti mi? | Görev 4: tam ölçüm, karne |

Lab 2'yi bitirdiyseniz aşağıdaki A/B testi **sizin yazdığınız** `taslak/v2.md`'yi ölçer.

### Önceki haftalardan
| Hafta | Ne yapmıştık | Bu notebook'ta ne oluyor |
|---|---|---|
| 1 | Test seti, metrikler, temel çizgi (`03_degerlendirme`) | Aynı test seti, iki **şablon sürümü** karşılaştırılıyor (A/B) |
| 2 | Önceki haftayla karşılaştırma, karışıklık matrisi (`03_olc_ve_tartis`) | Metriklerin göremediği **üslup** için hakem (LLM-as-judge) |
| 3 | JSONL izleme, adım maliyeti | İnsan onayındaki red gerekçesi kayda geçiyor ve **test seti adayı** oluyor |

### Bu notebook'un konuları
| Bölüm | Konu | Projede nerede |
|---|---|---|
| 1 | A/B testi ve tekrarlanabilirlik | `eval/ab_testi.py` (`--tekrar`) |
| 2 | Hakem (LLM-as-judge) ve hakemin sınanması | `eval/hakem.py`, `eval/hakem_sinama.csv` |
| 3 | Kalite geri bildirimi | `geri_bildirim.py`, `eval/aday_uret.py` |
| 4 | Tam setteki sonuç ve karne | `eval/calistir.py`, `eval/karne.md` |

Her kod hücresinin üstünde iki cümle var: **Ne yapıyoruz?** ve **Neye bakacağız?** Hücreyi çalıştırmadan önce okuyun.

**Ne yapıyoruz?** Notebook'u projenin koduna bağlıyor, talebi ve modeli seçiyoruz.
**Neye bakacağız?** Son satırda model ve talep yazmalı. Hata varsa önce `python kontrol.py` çalıştırın.

In [ ]:
import csv, os, sys, textwrap
from itertools import zip_longest
from pathlib import Path

PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))   # asistan paketi
sys.path.insert(0, str(PROJE_KOKU / "eval"))  # hakem.py, aday_uret.py

MODEL = "bulut"     # "yerel" de çalışır; küçük modelde sonuçlar daha değişken olur
TALEP = "Kartımdan aidat iki kez kesildi"
MUSTERI = "M0007"


def yan_yana(sol_baslik, sol, sag_baslik, sag, genislik=52):
    """İki metni iki sütun halinde yazar: solda önceki hal, sağda yeni hal."""
    sol_satir = [s for p in sol.split("\n") for s in (textwrap.wrap(p, genislik) or [""])]
    sag_satir = [s for p in sag.split("\n") for s in (textwrap.wrap(p, genislik) or [""])]
    print(f"{sol_baslik:<{genislik}} │ {sag_baslik}\n{'─' * genislik}─┼─{'─' * genislik}")
    for a, b in zip_longest(sol_satir, sag_satir, fillvalue=""):
        print(f"{a:<{genislik}} │ {b}")


print(f"Model: {MODEL} | Talep: {TALEP!r} ({MUSTERI})")

## 1. A/B testi ve tekrarlanabilirlik

A/B testi: aynı talepler, iki şablon sürümü, aynı metrikler. Değişken **tek**: şablon.
Model olasılıksal çalışır; aynı sürümün iki koşusu bile birebir aynı değildir. Bu yüzden her sürümü **birkaç kez** çalıştırırız
(`eval/ab_testi.py --tekrar 2`). Fark, koşular arasındaki oynamadan büyük değilse "fark yok" deriz.

**Ne yapıyoruz?** Küçük bir A/B testi: aynı talebi taslak **v1** (Hafta 3 metni) ve **v2** ile ikişer kez çalıştırıyoruz.
Sürüm kodla değil ortam değişkeniyle seçiliyor (`SABLON_TASLAK`, Notebook 01 Bölüm 4).
**Neye bakacağız?** Davranışlar aynı mı, yanıtlar nasıl farklı? Aynı sürümün iki koşusu da farklı olabilir.

In [ ]:
from asistan import cevapla

ab = {}
for surum in ("v1", "v2"):
    os.environ["SABLON_TASLAK"] = surum          # sürüm .env/ortamdan seçilir; kod değişmez
    ab[surum] = [cevapla(TALEP, MUSTERI, MODEL) for _ in range(2)]
os.environ.pop("SABLON_TASLAK")

for surum, kosular in ab.items():
    print(f"taslak {surum}: davranışlar = {[k['davranis'] for k in kosular]}")
print()
yan_yana("ÖNCE: taslak v1 (Hafta 3 metni)", ab["v1"][0]["yanit"], "SONRA: taslak v2", ab["v2"][0]["yanit"])

## 2. Hakem (LLM-as-judge)

Davranış doğruluğu "doğru **karar**"ı ölçer. İki sürüm de aynı kararı veriyorsa fark **üslupta**dır ve metrikler bunu göremez.
**Hakem** (`eval/hakem.py`) güçlü bir modeldir (`CLAUDE_MODEL_GUCLU`). Yanıtları dört ölçütte 1–5 arası puanlar
(doğruluk, eksiksizlik, banka dili, empati) ve her puanın gerekçesini yazar. Şablonu da `prompts/hakem/v1.md`'dedir.

**Ne yapıyoruz?** İki sürümün ilk yanıtını hakeme puanlatıyoruz. Hakem bulut model ister; yoksa hücre bunu söyler ve atlar.
**Neye bakacağız?** Hangi ölçütte fark var ve hakemin gerekçesi bu farkı açıklıyor mu? Önce **siz** puanlayın, sonra hakeme bakın.

In [ ]:
from hakem import OLCUTLER, puanla

puanlar = {surum: puanla({"id": surum, "talep": TALEP, "tahmin_davranis": k[0]["davranis"], "yanit": k[0]["yanit"],
                          "kaynaklar": ";".join(k[0]["kaynaklar"]),
                          "not": "M0007'de 2026-09-03'te iki aidat (I00287, I00288); iade önerisi I00288 olmalı"})
           for surum, k in ab.items()}
if None in puanlar.values():
    print("Hakem çalışmadı: CLAUDE_MODEL_GUCLU ve bulut anahtarı gerekli. Eğitmenin sonucu aşağıdaki tabloda.")
else:
    print(f"{'ölçüt':<13}{'v1':>4}{'v2':>4}   v2 gerekçesi")
    for o in OLCUTLER:
        print(f"{o:<13}{getattr(puanlar['v1'], o):>4}{getattr(puanlar['v2'], o):>4}   {getattr(puanlar['v2'], o + '_gerekce')[:70]}")

**Hakem de bir modeldir ve yanılabilir.** Bu yüzden ona güvenmeden önce **sınarız**: `eval/hakem_sinama.csv` dosyasında 10 yanıt
ve eğitmenin elle verdiği puanlar var. `python eval/hakem.py --sinama` hakemi aynı yanıtlarda çalıştırıp karşılaştırır.
Eğitmenin koşusu:

| Ölçüt | Tam eşleşme | ±1 içinde | Ortalama fark (hakem − eğitmen) |
|---|---|---|---|
| doğruluk | %40 | %70 | −0,7 |
| eksiksizlik | %40 | %90 | −0,3 |
| banka dili | %30 | %100 | −0,5 |
| empati | %20 | %100 | 0,0 |

Hakem **kötüyü iyiden ayırıyor**: yapılmamış işlem uyduran ve prosedüre aykırı söz veren yanıtlar 1 aldı.
Ama doğrulukta sistematik olarak sert: hakeme müşteri verisi verilmiyor, veriden gelen doğru bilgiyi (kartın son 4 hanesi) uydurma sayıyor.
Sonuç: banka dili ve empatide A/B kararına **destek** olarak kullanılabilir; doğrulukta tek başına değil.

## 3. Kalite geri bildirimi: sahadaki hata, bir sonraki ölçümün sorusu

Test seti bizim tahmin ettiğimiz durumlardan oluşur. Sahada **temsilci** ise tahmin etmediğimiz hataları görür.
İnsan onayında temsilci bir taslağı reddeder ya da düzeltirse gerekçesi `eval/geri_bildirim.csv`'ye yazılır (maskeli).
`eval/aday_uret.py` bu satırları **test seti adayına** çevirir. Aday test setine kendiliğinden girmez: **eğitmen** inceler ve onaylar.
Test beklentileri yalnızca eğitmen onayıyla değişir.

```
insan onayı ──► red / düzeltme + gerekçe ──► eval/geri_bildirim.csv ──► aday_uret.py ──► test_seti_adaylari.csv ──► (eğitmen) test_seti.csv
```

**Ne yapıyoruz?** A/B'deki koşulardan insan onayı bekleyenlerden birini **temsilci gibi** reddediyoruz ve bir gerekçe yazıyoruz (`ajan.onayla`).
`demo.py`'de "h" deyince olan budur.
**Neye bakacağız?** Talep temsilciye gider ve gerekçe `geri_bildirim.csv`'nin son satırında görünür.

In [ ]:
from asistan import onayla

bekleyen = next((k for kosular in ab.values() for k in kosular if k["onay_bekliyor"]), None)
if bekleyen is None:
    print("Bu çalıştırmada insan onayı bekleyen koşu yok (hepsi başka bir davranışa gitti). Bölüm 1'i tekrar çalıştırın.")
else:
    GEREKCE = "Yanıt iadenin ne zaman yansıyacağını söylemiyor; prosedürde süre yazıyor."
    red = onayla(bekleyen["oturum_id"], False, "egitmen", gerekce=GEREKCE)
    print("Sonuç:", red["davranis"], "|", red["adimlar"][-1])
    with open(PROJE_KOKU / "eval" / "geri_bildirim.csv", encoding="utf-8") as f:
        son = list(csv.DictReader(f))[-1]
    for alan in ("zaman", "talep", "karar", "gerekce", "onaylayan"):
        print(f"  {alan:<10} {son[alan]}")

**Ne yapıyoruz?** Biriken geri bildirimleri test seti adayına çeviriyoruz (`python eval/aday_uret.py` ile aynı).
**Neye bakacağız?** Adayın beklenen davranışı bir **öneri** (`red` → `temsilciye_aktar`). Bu talep için doğru beklenti bu mu?
Kategori ve kaynak boş: onları eğitmen doldurur.

In [ ]:
import aday_uret

cikis_kodu = aday_uret.main()   # 0 = sorun yok

## 4. Tam setteki sonuç ve karne

Tek talep bir A/B testi değildir; karar 37 satırlık test setinde verilir. Eğitmenin koşusunda (taslak v1 ve v2, 2'şer tekrar):

| Metrik (ortalama, 2 koşu) | Bulut v1 | Bulut v2 | Yerel v1 | Yerel v2 |
|---|---|---|---|---|
| Davranış doğruluğu | %93 | **%97** | %58 | **%70** |
| Olması gereken bilgi | %90 | **%98** | %55 | **%67** |
| Kişisel veri (modele / yanıtta) | 0 / 0 | 0 / 0 | 0 / 0 | 0 / 0 |
| Ortalama süre | 15,9 sn | 13,2 sn | 22,8 sn | 19,3 sn |
| Hakem: banka dili / empati (1-5) | 3,3 / 3,1 | **3,9 / 4,0** | – | – |

Bulutta v2 kararda biraz önde (+4); asıl farkı **içerik** ve **hakem** gösteriyor. Yerelde v2 küçük modele açıkça yardım ediyor.
Bu yüzden v2 varsayılan sürüm oldu.

**Ne yapıyoruz?** `eval/karne.md`'den test setinin haftalık sonuçlarını okuyoruz (`python eval/karne.py` üretir).
**Neye bakacağız?** Her haftanın eklediği katman hangi metriği değiştirdi? Hafta 4'te davranış ve **kişisel veri** satırlarına bakın.

In [ ]:
import re

karne = (PROJE_KOKU / "eval" / "karne.md").read_text(encoding="utf-8")
for baslik in ("### test_seti, bulut model", "### test_seti, yerel model"):
    satirlar = [s for s in karne.split(baslik)[1].split("###")[0].splitlines() if re.match(r"\| Hafta \d", s)]
    print(baslik[4:])
    print(f"  {'hafta':<9}{'davranış':>10}{'kaynak':>9}   kişisel veri (modele / yanıtta)")
    for s in satirlar:
        h = [x.strip() for x in s.strip("|").split("|")]   # Hafta, katman, satır, kategori, davranış, kaynak, gizli, kişisel, süre
        print(f"  {h[0]:<9}{h[4]:>10}{h[5].split()[0]:>9}   {h[7]}")
    print()

Tam ölçümü Görev 4'te siz yapacaksınız: `python eval/calistir.py` (yerelde ~20 dk), sonra `eval/karne.md`.

---

## 💬 Tartışma
1. A/B testinde v2 bir metrikte iyileşip başka birinde kötüleşirse hangisine bakarak karar verirsiniz?
   Hakem puanı ile test seti metriği çelişirse hangisine güvenirsiniz? Hakeme güvenmeden önce ne yaptık?
2. Bölüm 3'teki aday test setine girmeli mi? Beklenen davranışı kim, neye göre belirler?
   Ölçüm kötüleşince hata her zaman sistemde midir, yoksa test de eskiyebilir mi?